In [1]:
# Project 2 – Data Cleaning, Missing Data, Transformations

In [2]:
# 1. Load the dataset

In [3]:
import pandas as pd 
import numpy as np

In [4]:

df = pd.read_csv("/Users/keerthanat/Desktop/marketing_campaign_dataset.csv")

In [5]:
# Display basic information to understand the structure of the data
df.shape
df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  object 
 2   Campaign_Type     200000 non-null  object 
 3   Target_Audience   200000 non-null  object 
 4   Duration          200000 non-null  object 
 5   Channel_Used      200000 non-null  object 
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  object 
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  object 
 10  Language          200000 non-null  object 
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  object 
 15  Date              200000 non-null  object 
dtypes: float64(2), int64

In [6]:
# 2) Keep only columns relevant to the research question
cols_used = [
    "Campaign_ID", "Company", "Campaign_Type", "Target_Audience",
    "Duration", "Channel_Used",
    "Conversion_Rate", "Acquisition_Cost", "ROI",
    "Clicks", "Impressions", "Engagement_Score",
    "Date"
]
df = df[cols_used].copy()

In [7]:
# Confirm we kept the correct columns
df.shape
df.columns

Index(['Campaign_ID', 'Company', 'Campaign_Type', 'Target_Audience',
       'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost',
       'ROI', 'Clicks', 'Impressions', 'Engagement_Score', 'Date'],
      dtype='object')

In [8]:
# 3) Check missing values (even if df.info() shows non-null)
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts


Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Clicks              0
Impressions         0
Engagement_Score    0
Date                0
dtype: int64

In [9]:
# Percent missing values per column 
missing_perc = (df.isna().mean() * 100).sort_values(ascending=False)
missing_perc

Campaign_ID         0.0
Company             0.0
Campaign_Type       0.0
Target_Audience     0.0
Duration            0.0
Channel_Used        0.0
Conversion_Rate     0.0
Acquisition_Cost    0.0
ROI                 0.0
Clicks              0.0
Impressions         0.0
Engagement_Score    0.0
Date                0.0
dtype: float64

In [12]:
# 4) Fix data types
# Duration is currently an object; convert to numeric so we can analyze it.
df["Duration"] = (
    df["Duration"]
    .astype(str)                          # make sure it's string
    .str.extract(r"(\d+)", expand=False)  # extract the number part
)

# Convert extracted values to numeric
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

# (Optional) Fill any missing values created by bad strings, then convert to int
df["Duration"] = df["Duration"].fillna(df["Duration"].median()).astype(int)
# Acquisition_Cost is currently an object (may include $ signs or commas).
df["Acquisition_Cost"] = (
    df["Acquisition_Cost"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Acquisition_Cost"] = pd.to_numeric(df["Acquisition_Cost"], errors="coerce")
# Convert Date from object to datetime so we can do time-based analysis later if needed.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
# Convert categorical variables to category type
cat_cols = ["Company", "Campaign_Type", "Target_Audience", "Channel_Used"]
for c in cat_cols:
    df[c] = df[c].astype("category")


In [13]:
df.dtypes

Campaign_ID                  int64
Company                   category
Campaign_Type             category
Target_Audience           category
Duration                     int64
Channel_Used              category
Conversion_Rate            float64
Acquisition_Cost           float64
ROI                        float64
Clicks                       int64
Impressions                  int64
Engagement_Score             int64
Date                datetime64[ns]
dtype: object

In [14]:
df["Duration"].head(10)

0    30
1    60
2    30
3    60
4    15
5    15
6    60
7    45
8    15
9    15
Name: Duration, dtype: int64

In [15]:
# 5) Handle missing values created by type conversion (if any)
df.isna().sum().sort_values(ascending=False)

Campaign_ID         0
Company             0
Campaign_Type       0
Target_Audience     0
Duration            0
Channel_Used        0
Conversion_Rate     0
Acquisition_Cost    0
ROI                 0
Clicks              0
Impressions         0
Engagement_Score    0
Date                0
dtype: int64

In [19]:
# For Acquisition_Cost: if missing values exist, impute with median
df["Acquisition_Cost"] = df["Acquisition_Cost"].fillna(df["Acquisition_Cost"].median())
# Drop rows missing key predictors/outcomes needed for analysis
df = df.dropna(subset=["Channel_Used", "Target_Audience", "Duration", "Conversion_Rate", "ROI"])

In [22]:
# 6) Remove invalid/impossible values (data cleaning)
df = df[df["Duration"] > 0]
df = df[(df["Clicks"] >= 0) & (df["Impressions"] >= 0) & (df["Engagement_Score"] >= 0)]
df = df[df["Acquisition_Cost"] >= 0]

In [23]:
df.shape

(200000, 13)

In [29]:
# 7) Data transformations (new variables that help analysis)
#Cost per click (CPC) as an additional efficiency metric
df["CPC"] = np.where(df["Clicks"] > 0, df["Acquisition_Cost"] / df["Clicks"], np.nan)
#Click-through rate (CTR)
df["CTR"] = np.where(df["Impressions"] > 0, df["Clicks"] / df["Impressions"], np.nan)
#Bin duration into categories
df["Duration_cat"] = pd.cut(
    df["Duration"],
    bins=[-np.inf, 7, 30, np.inf],
    labels=["Short", "Medium", "Long"]
)

In [30]:
# 8) Create dummy variables for categorical predictors (Channel_Used, Target_Audience)
dummies = pd.get_dummies(df[["Channel_Used", "Target_Audience"]], drop_first=True)
df = pd.concat([df, dummies], axis=1)


In [31]:
df.shape
df.head()
df.isna().sum().sort_values(ascending=False)
df.describe(include="all")


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Clicks,...,Duration_cat,Channel_Used_Facebook,Channel_Used_Google Ads,Channel_Used_Instagram,Channel_Used_Website,Channel_Used_YouTube,Target_Audience_Men 18-24,Target_Audience_Men 25-34,Target_Audience_Women 25-34,Target_Audience_Women 35-44
count,200000.000000,200000,200000,200000,200000.000000,200000,200000.000000,200000.000000,200000.000000,200000.000000,...,200000,200000,200000,200000,200000,200000,200000,200000,200000,200000
unique,NaN,5,5,5,NaN,6,NaN,NaN,NaN,NaN,...,2,2,2,2,2,2,2,2,2,2
top,NaN,TechCorp,Influencer,Men 18-24,NaN,Email,NaN,NaN,NaN,NaN,...,Medium,False,False,False,False,False,False,False,False,False
freq,NaN,40237,40169,40258,NaN,33599,NaN,NaN,NaN,NaN,...,100034,167181,166562,166608,166640,166608,159742,159977,159987,160313
mean,100000.500000,NaN,NaN,NaN,37.503975,NaN,0.080070,12504.393040,5.002438,549.772030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,15.000000,NaN,0.010000,5000.000000,2.000000,100.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,50000.750000,NaN,NaN,NaN,30.000000,NaN,0.050000,8739.750000,3.500000,325.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,100000.500000,NaN,NaN,NaN,30.000000,NaN,0.080000,12496.500000,5.010000,550.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,150000.250000,NaN,NaN,NaN,45.000000,NaN,0.120000,16264.000000,6.510000,775.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,200000.000000,NaN,NaN,NaN,60.000000,NaN,0.150000,20000.000000,8.000000,1000.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
